In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import os
import json
import joblib

# --- 1. Load & Preprocess Data ---
file_path = 'smart_traffic_management_dataset.csv'

if not os.path.exists(file_path):
    print(f"Error: The file '{file_path}' was not found.")
    exit(1)

try:
    df = pd.read_csv(file_path)
    print("Dataset loaded successfully.")
    print("\nFirst 5 rows of the dataset:")
    print(df.head())

    # Feature Engineering: Extract hour from timestamp
    df['hour'] = pd.to_datetime(df['timestamp']).dt.hour
    df.drop(columns=['timestamp'], inplace=True)

    # Encode categorical columns (one-hot encoding)
    categorical_cols = ['location_id', 'weather_condition', 'signal_status']
    df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

    # Normalize numerical features
    numerical_cols = ['traffic_volume', 'avg_vehicle_speed', 'vehicle_count_cars', 
                     'vehicle_count_trucks', 'vehicle_count_bikes', 'temperature', 'humidity', 'hour']
    scaler = StandardScaler()
    df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

    # Separate features and target
    X = df.drop('accident_reported', axis=1)
    y = df['accident_reported']

    # Save feature columns for prediction
    feature_columns = X.columns.tolist()
    with open('plots/feature_columns.json', 'w') as f:
        json.dump(feature_columns, f, indent=4)

    # Check class distribution
    print("\nClass distribution of accident_reported:")
    print(y.value_counts(normalize=True))

    # --- 2. Split Data into Training and Testing Sets ---
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    print("\nTraining and testing data split successfully.")
    print(f"X_train shape: {X_train.shape}")
    print(f"X_test shape: {X_test.shape}")

    # Handle class imbalance with SMOTE
    smote = SMOTE(random_state=42)
    X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)
    print("\nClass distribution after SMOTE:")
    print(pd.Series(y_train_balanced).value_counts())

    # --- 3. Train Random Forest Model with Hyperparameter Tuning ---
    rf = RandomForestClassifier(random_state=42, class_weight='balanced')
    param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [10, 15, 20],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2]
    }
    grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
    print("\nPerforming grid search for hyperparameter tuning...")
    grid_search.fit(X_train_balanced, y_train_balanced)
    clf = grid_search.best_estimator_
    print(f"Best parameters: {grid_search.best_params_}")
    print(f"Best cross-validation accuracy: {grid_search.best_score_:.2%}")

    # Save the model and scaler
    os.makedirs('plots', exist_ok=True)
    joblib.dump(clf, 'plots/model.joblib')
    joblib.dump(scaler, 'plots/scaler.joblib')
    print("\nModel and scaler saved to 'plots/model.joblib' and 'plots/scaler.joblib'.")

    # Save best parameters
    with open('plots/best_params.json', 'w') as f:
        json.dump(grid_search.best_params_, f, indent=4)

    # --- 4. Predict & Evaluate ---
    y_pred = clf.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print("\n--- Model Evaluation ---")
    print(f'Model Accuracy: {accuracy*100:.2f}%')

    # Generate classification report
    class_report = classification_report(y_test, y_pred, output_dict=True)
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    # --- 5. Visual Outputs ---
    os.makedirs('plots', exist_ok=True)

    # 5a. Confusion Matrix Heatmap
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix Heatmap')
    plt.xlabel('Predicted Label')
    plt.ylabel('Actual Label')
    plt.savefig('plots/confusion_matrix.png')
    plt.close()

    # 5b. Feature Importance Bar Graph
    feat_importances = pd.Series(clf.feature_importances_, index=X_train.columns).nlargest(10)
    plt.figure(figsize=(8, 6))
    sns.barplot(x=feat_importances, y=feat_importances.index, color='green')
    plt.title('Top 10 Feature Importances')
    plt.xlabel('Importance Score')
    plt.gca().invert_yaxis()
    plt.savefig('plots/feature_importance.png')
    plt.close()

    # 5c. Feature Correlation Heatmap
    plt.figure(figsize=(12, 10))
    corr = X.corr()
    sns.heatmap(corr, cmap='coolwarm', square=True, linewidths=0.5)
    plt.title('Feature Correlation Heatmap')
    plt.savefig('plots/correlation_heatmap.png')
    plt.close()

    # 5d. Chart.js Bar Chart for Performance Metrics
    chartjs_data = {
        "type": "bar",
        "data": {
            "labels": ["Accuracy", "Precision (0)", "Recall (0)", "F1-Score (0)", 
                       "Precision (1)", "Recall (1)", "F1-Score (1)"],
            "datasets": [{
                "label": "Model Performance Metrics",
                "data": [
                    accuracy,
                    class_report['0']['precision'],
                    class_report['0']['recall'],
                    class_report['0']['f1-score'],
                    class_report['1']['precision'],
                    class_report['1']['recall'],
                    class_report['1']['f1-score']
                ],
                "backgroundColor": ["#2ecc71", "#3498db", "#3498db", "#3498db", 
                                   "#e74c3c", "#e74c3c", "#e74c3c"],
                "borderColor": ["#27ae60", "#2980b9", "#2980b9", "#2980b9", 
                                "#c0392b", "#c0392b", "#c0392b"],
                "borderWidth": 1
            }]
        },
        "options": {
            "responsive": True,
            "scales": {
                "y": {
                    "beginAtZero": True,
                    "title": {"display": True, "text": "Score"},
                    "max": 1
                },
                "x": {
                    "title": {"display": True, "text": "Metric"}
                }
            },
            "plugins": {
                "legend": {"display": True},
                "title": {"display": True, "text": f"Model Performance Metrics (Accuracy: {accuracy:.2%})"}
            }
        }
    }

    # Save Chart.js configuration
    with open('plots/performance_metrics_chartjs.json', 'w') as f:
        json.dump(chartjs_data, f, indent=4)
    print("\nChart.js configuration saved to 'plots/performance_metrics_chartjs.json'.")

except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please ensure it is in the same directory.")
except Exception as e:
    print(f"An error occurred: {e}")


Dataset loaded successfully.

First 5 rows of the dataset:
             timestamp  location_id  traffic_volume  avg_vehicle_speed  \
0  2024-01-01 00:00:00            4             504          53.124162   
1  2024-01-01 00:01:00            5             209          44.947850   
2  2024-01-01 00:02:00            3             572          63.179229   
3  2024-01-01 00:03:00            5             699          42.269697   
4  2024-01-01 00:04:00            5             639          72.185791   

   vehicle_count_cars  vehicle_count_trucks  vehicle_count_bikes  \
0                 142                    24                   44   
1                 862                    50                   23   
2                 317                    12                   10   
3                 709                    43                   21   
4                 594                    34                   14   

  weather_condition  temperature   humidity  accident_reported signal_status  
0       